# TP2-1 — Création d'un agent IA

Durée indicative : 45 min.

**La tâche** : le même assistant juridique qu'au TP1, mais cette fois **c'est lui qui décide** quand chercher dans la loi et quand calculer. 

**La méthode** : un agent, trois écritures, les mêmes questions, les mêmes mesures :

| Étape | Écriture | Ce qu'on apprend |
|---|---|---|
| 1 | La boucle ReAct à la main | Il n'y a pas de magie : un `while` |
| 2 | `create_agent` | La même boucle, prête à l'emploi |
| 3 | Un graphe LangGraph explicite | La même boucle, visible et bornée |

Le banc (`src/banc_agent.py`) compte, pour chaque question : appels au modèle, appels d'outils, tokens, réponse juste, limite atteinte.

### Comment compléter

Deux `TODO` en texte (le profil de l'agent, puis en bonus la description d'un outil) et un `TODO` d'une ligne de code (la condition d'arrêt du graphe), avec l'indice à côté.

In [ ]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(RACINE), str(RACINE / "02-agent")]

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

from common.llm import get_llm, get_provider, invoquer
from common.rag import indexer_corpus, questions_test
from common.tools import outils_de_base
from common.viz import afficher
from src.banc_agent import BancAgent, afficher_trace

CORPUS = "loi-repost"

llm = get_llm()
outils = outils_de_base()  # rechercher_corpus, calculatrice
OUTILS = {o.name: o for o in outils}
print(f"{indexer_corpus(CORPUS)} passages indexés · fournisseur {get_provider()}")

questions = questions_test(CORPUS, "questions_agent.json")
banc = BancAgent(questions)
for q in questions:
    print("-", q["question"], "→", q["faits_attendus"], "· outils attendus :", q["outils_attendus"])

Trois questions : deux demandent une recherche puis un calcul, la troisième un calcul seul. On regardera si l'agent appelle les bons outils, dans le bon ordre, et s'il s'arrête.

### Ce que le modèle voit d'un outil

Un outil, pour le modèle, c'est un nom, une description et un schéma d'arguments. Rien d'autre. La description est un prompt.

In [ ]:
for o in outils:
    print(f"{o.name}({', '.join(o.args)})")
    print("   ", o.description.replace("\n", "\n    "))
    print()

## Étape 1 — La boucle ReAct à la main

Le modèle reçoit les outils (`bind_tools`). À chaque tour, soit il répond en texte, soit il renvoie un ou plusieurs **appels d'outil** (`tool_calls`). Le programme exécute, renvoie le résultat dans un `ToolMessage`, et le modèle continue.

Le profil de l'agent est le `TODO`. Il devient le message système. Il dit au modèle qui il est, quels outils utiliser et quand, et comment répondre.

In [ ]:
# ---- TODO : le profil de l'agent, en texte -------------------------------------------
# Rôle et public ; quand appeler rechercher_corpus (toute question sur la loi) ; quand
# appeler calculatrice (tout calcul, jamais de tête) ; forme de la réponse (une ou deux
# phrases, les chiffres exacts) ; ce qu'il ne doit jamais faire (inventer un montant).
PROFIL = """..."""
# ---- fin TODO ------------------------------------------------------------------------

llm_avec_outils = llm.bind_tools(outils)


def react_a_la_main(question: str, max_tours: int = 8) -> list:
    """La boucle ReAct. Retourne tous les messages échangés."""
    messages = [SystemMessage(PROFIL), HumanMessage(question)]
    for _ in range(max_tours):
        reponse = invoquer(llm_avec_outils, messages)
        messages.append(reponse)
        if not reponse.tool_calls:  # pas d'appel d'outil : c'est la réponse finale
            break
        for appel in reponse.tool_calls:
            resultat = OUTILS[appel["name"]].invoke(appel["args"])
            messages.append(ToolMessage(str(resultat), tool_call_id=appel["id"]))
    return messages


mesures = banc.mesurer("1 - ReAct à la main", react_a_la_main)

À observer :

- **Les outils appelés** et leur ordre. Attendu : recherche puis calcul pour les deux premières questions, calcul seul pour la troisième.
- **La reformulation** : l'agent ne transmet pas la question telle quelle à `rechercher_corpus`, il la réécrit. S'il remplace « contribue à l'organisation » par « participe », il trouve un autre article et un autre montant. Une reformulation, c'est une décision.
- **L'arrêt** : `max_tours` est la seule chose qui empêche la boucle de tourner sans fin.

La trace complète d'une question, message par message :

In [ ]:
afficher_trace(banc.traces["1 - ReAct à la main"][1])

## Étape 2 — `create_agent`

La même boucle, fournie par LangChain 1.x. Trois arguments : le modèle, les outils, le profil. Le résultat est déjà un graphe LangGraph compilé : la boucle est devenue deux nœuds (`model`, `tools`) et une arête conditionnelle.

`create_agent` remplace `create_react_agent` des tutoriels antérieurs à LangChain 1.x. Il accepte en plus un checkpointer (séquence 2b) et des *middlewares* pour intervenir avant ou après le modèle.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, outils, system_prompt=PROFIL)


def lancer_create_agent(question: str) -> list:
    resultat = agent.invoke({"messages": [HumanMessage(question)]}, config={"recursion_limit": 16})
    return resultat["messages"]


mesures = banc.mesurer("2 - create_agent", lancer_create_agent)
afficher(agent)

Mêmes questions, mêmes outils, même profil : les chiffres doivent être proches de l'étape 1. Ce qui change, c'est ce qu'on ne voit pas : la boucle est un graphe, donc dessinable, bornée par `recursion_limit`, et prête à recevoir un checkpointer.

Une différence de comptage possible : `create_agent` exécute les appels d'outils d'un même tour en parallèle, la boucle à la main les exécute l'un après l'autre. Le nombre d'appels au modèle peut donc différer d'un.

## Étape 3 — Le graphe explicite

Cette fois, la boucle est écrite en LangGraph, nœud par nœud :

- **État** : la liste des messages, avec le reducer `add_messages` (un nœud retourne un delta, LangGraph l'ajoute).
- **Nœud `agent`** : un appel au modèle avec les outils.
- **Nœud `outils`** : `ToolNode` exécute les appels d'outils du dernier message et produit les `ToolMessage`. Ce sont les quatre lignes du `for` de l'étape 1.
- **Arête conditionnelle** : après `agent`, aller vers `outils` s'il y a des appels d'outils, sinon `END`. C'est le `if not reponse.tool_calls: break` de l'étape 1.
- **`recursion_limit`** : chaque passage dans un nœud compte un pas. Au-delà, `GraphRecursionError`.

Le `TODO` est la condition d'arrêt : une ligne.

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


class Etat(TypedDict):
    messages: Annotated[list, add_messages]


def noeud_agent(etat: Etat) -> dict:
    reponse = invoquer(llm_avec_outils, [SystemMessage(PROFIL), *etat["messages"]])
    return {"messages": [reponse]}


def routage(etat: Etat) -> str:
    """Après le nœud agent : "outils" s'il reste des appels d'outils à exécuter, sinon END."""
    dernier = etat["messages"][-1]
    # ---- TODO : une ligne ---------------------------------------------------------------
    # Indice : un AIMessage a un attribut .tool_calls (liste, vide s'il n'y a pas d'appel).
    # Tant que cette ligne renvoie END, le graphe ne passe jamais par les outils.
    return END
    # ---- fin TODO -----------------------------------------------------------------------


LIMITE_RECURSION = 16  # 2 pas par tour (agent + outils) : 16 pas = 8 tours au maximum

constructeur = StateGraph(Etat)
constructeur.add_node("agent", noeud_agent)
constructeur.add_node("outils", ToolNode(outils))
constructeur.add_edge(START, "agent")
constructeur.add_conditional_edges("agent", routage, {"outils": "outils", END: END})
constructeur.add_edge("outils", "agent")
graphe = constructeur.compile()

afficher(graphe)


def lancer_graphe(question: str) -> list:
    resultat = graphe.invoke(
        {"messages": [HumanMessage(question)]}, config={"recursion_limit": LIMITE_RECURSION}
    )
    return resultat["messages"]


mesures = banc.mesurer("3 - graphe explicite", lancer_graphe)

### Vérifier que la limite fonctionne

Un agent qui boucle coûte de l'argent à chaque pas. La limite doit couper net. Ici, une limite volontairement trop basse pour une question qui demande deux outils : le graphe s'arrête avec une erreur propre, attrapable.

In [ ]:
from langgraph.errors import GraphRecursionError

try:
    graphe.invoke({"messages": [HumanMessage(questions[0]["question"])]}, config={"recursion_limit": 3})
except GraphRecursionError as e:
    print("Arrêt propre :", str(e)[:120], "…")

## Le tableau final

In [ ]:
banc.tableau()

Trois écritures, les mêmes chiffres à peu de chose près : c'est la même boucle.

| Écriture | Ce qu'elle apporte | Quand |
|---|---|---|
| À la main | On voit tout, on comprend tout | Pour apprendre, pour un script jetable |
| `create_agent` | Trois lignes, graphe compilé, checkpointer et middlewares prêts | Le point de départ d'un vrai projet |
| Graphe explicite | Chaque nœud, chaque arête sous contrôle ; on peut en ajouter | Dès qu'il faut un nœud de plus : résumé, validation humaine, routage, sous-graphe |

### Ce que le banc montre aussi

- Les **tokens** : chaque tour renvoie tout l'historique. Deux outils, c'est trois appels au modèle, et le troisième porte tout ce qui précède. Le coût d'un agent n'est pas linéaire dans le nombre d'étapes.
- Les **erreurs de l'agent** ne sont pas des erreurs de code : une question reformulée, un outil appelé deux fois, un calcul fait de tête malgré la consigne (une ligne à 2/3 dans la colonne « outils attendus » : le modèle a multiplié 3 × 3 750 lui-même). D'une exécution à l'autre, ce n'est pas la même écriture qui fait l'erreur : le modèle n'est pas déterministe, même à température 0. Le profil et la description des outils sont les seuls leviers, et ils ne garantissent rien. Le petit modèle gratuit le montre plus qu'un grand.

### Ce qui reste

- L'agent oublie tout entre deux questions, et un crash perd tout : **checkpointer et threads**, séquence 2b.
- Il exécute ce qu'il décide, sans demander : **validation humaine** avec `interrupt`, séquence 2b.
- Ses outils sont locaux : **MCP** pour en consommer à distance, séquence 2b.

## Bonus

**La description d'un outil est un prompt.** Le nom `calculatrice` en dit déjà long : le modèle l'appelle même si sa description est vide. Avec un nom opaque, seule la description compte. Ici, un outil `op_17` qui fait la même chose que la calculatrice. Sans description utile, le modèle calcule de tête ; avec, il l'appelle.

In [ ]:
from langchain_core.tools import tool
from common.tools import calculatrice, rechercher_corpus

# ---- TODO : la description, en texte -------------------------------------------------
# D'abord relancer avec "Outil." et regarder la colonne « outils attendus ». Puis écrire
# une description qui dit ce que fait l'outil, sur quoi, avec un exemple, et quand l'appeler.
DESCRIPTION_OP_17 = "Outil."
# ---- fin TODO ------------------------------------------------------------------------


@tool(description=DESCRIPTION_OP_17)
def op_17(x: str) -> str:
    return calculatrice.invoke({"expression": x})


outils_bonus = [rechercher_corpus, op_17]
llm_avec_outils = llm.bind_tools(outils_bonus)
graphe_bonus = StateGraph(Etat)
graphe_bonus.add_node("agent", noeud_agent)
graphe_bonus.add_node("outils", ToolNode(outils_bonus))
graphe_bonus.add_edge(START, "agent")
graphe_bonus.add_conditional_edges("agent", routage, {"outils": "outils", END: END})
graphe_bonus.add_edge("outils", "agent")
graphe_bonus = graphe_bonus.compile()

questions_bonus = [q | {"outils_attendus": ["op_17" if o == "calculatrice" else o for o in q["outils_attendus"]]} for q in questions]
banc_bonus = BancAgent(questions_bonus)
banc_bonus.mesurer(
    f"bonus - op_17 : {DESCRIPTION_OP_17[:25]}",
    lambda q: graphe_bonus.invoke({"messages": [HumanMessage(q)]}, config={"recursion_limit": LIMITE_RECURSION})["messages"],
)
banc_bonus.tableau()

llm_avec_outils = llm.bind_tools(outils)  # remettre en état

**Sans reducer.** Retirer `Annotated[..., add_messages]` de `Etat` (laisser `messages: list`), recompiler, relancer une question : le nœud `agent` écrase l'historique à chaque passage, le `ToolMessage` arrive sans son appel, le modèle ne comprend plus rien. Remettre le reducer.